> **Nota de actualización.** Los datos completos 2009–2024 viven en `DatosAbiertos/` y el pipeline canónico está en `src/` (`descargar_datos.py`, `preparar_datos.py`, `preparar_campanas.py`, `entrenar_modelo.py`). Este notebook se conserva como exploración y usa rutas actualizadas; para reproducir el modelo final ejecuta `python src/entrenar_modelo.py`.


# Fase 3 — Modelo de riesgo con toda la historia disponible

Se utilizan los seis cortes electorales disponibles (2009, 2012, 2015, 2018, 2021 y 2024). Las elecciones 2009–2021 se integran como variables históricas de cada sección y 2024 es el objetivo. La evaluación separa municipios completos para medir generalización territorial.

El índice es territorial: orienta prioridades de intervención y no estima conducta individual ni causalidad.


## Metodología

1. Agregar cada elección a nivel de sección leyendo los 32 archivos por bloques.
2. Integrar para cada sección las variables de 2009, 2012, 2015, 2018 y 2021.
3. Entrenar y evaluar un Gradient Boosting sobre la abstención observada en 2024, reservando municipios completos.
4. Ajustar el modelo final con todas las secciones 2024 y generar el riesgo.
5. Simular X1 y X2 sobre las secciones Alto/Muy alto.


In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings('ignore', category=FutureWarning)
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
YEARS = [2009, 2012, 2015, 2018, 2021, 2024]
DATA_DIRS = {year: BASE_DIR / 'DatosAbiertos' / f'DatosAbiertos_DECEyEC_ConteosCensales{year}' for year in YEARS}
KEYS = ['EDOCVE', 'MPIOCVE', 'SECCION']
RAW_USECOLS = ['EDOCVE', 'MPIOCVE', 'SECCION', 'TIPOSEC', 'SEXO', 'EDAD', 'LN', 'SV', 'NV', 'NS']
CHUNK_SIZE = 250_000
for year, path in DATA_DIRS.items():
    files = sorted(path.glob(f'datosabiertos_deceyec_conteoscensales{year}_*.csv'))
    print(f'{year}: {len(files)} archivos')
    if len(files) != 32: raise FileNotFoundError(f'Se esperaban 32 archivos para {year}, se encontraron {len(files)}')


2009: 32 archivos
2012: 32 archivos
2015: 32 archivos
2018: 32 archivos
2021: 32 archivos
2024: 32 archivos


In [2]:
def agregar_eleccion(year):
    parciales = []
    files = sorted(DATA_DIRS[year].glob(f'datosabiertos_deceyec_conteoscensales{year}_*.csv'))
    for file in files:
        for chunk in pd.read_csv(file, encoding='utf-8-sig', usecols=RAW_USECOLS, chunksize=CHUNK_SIZE):
            chunk.columns = [str(c).replace('\ufeff', '').strip() for c in chunk.columns]
            for col in ['EDOCVE', 'MPIOCVE', 'SECCION', 'SEXO', 'EDAD', 'LN', 'SV', 'NV', 'NS']:
                chunk[col] = pd.to_numeric(chunk[col], errors='coerce').fillna(0)
            chunk['edad_grupo'] = pd.cut(chunk['EDAD'], bins=[17, 29, 59, np.inf], labels=['18_29', '30_59', '60_mas']).astype('object').fillna('sin_dato')
            base = chunk.groupby(KEYS, as_index=False, dropna=False).agg(LN=('LN', 'sum'), SV=('SV', 'sum'), NV=('NV', 'sum'), NS=('NS', 'sum'), TIPOSEC=('TIPOSEC', 'first'))
            edad = chunk.pivot_table(index=KEYS, columns='edad_grupo', values='LN', aggfunc='sum', fill_value=0, observed=False).reset_index()
            edad.columns.name = None
            for name in ['18_29', '30_59', '60_mas', 'sin_dato']:
                if name not in edad.columns: edad[name] = 0
            sexo = chunk.assign(sexo_nombre=chunk['SEXO'].map({0: 'hombres', 1: 'mujeres', 2: 'no_binario'}).fillna('sin_dato')).pivot_table(index=KEYS, columns='sexo_nombre', values='LN', aggfunc='sum', fill_value=0, observed=False).reset_index()
            sexo.columns.name = None
            for name in ['hombres', 'mujeres', 'no_binario', 'sin_dato']:
                if name not in sexo.columns: sexo[name] = 0
            parciales.append(base.merge(edad, on=KEYS, how='left').merge(sexo, on=KEYS, how='left'))
    result = pd.concat(parciales, ignore_index=True)
    numeric_cols = ['LN', 'SV', 'NV', 'NS', '18_29', '30_59', '60_mas', 'sin_dato_x', 'hombres', 'mujeres', 'no_binario', 'sin_dato_y']
    numeric_cols = [c for c in numeric_cols if c in result.columns]
    result[numeric_cols] = result[numeric_cols].fillna(0)
    result = result.groupby(KEYS, as_index=False).agg({c: 'sum' for c in numeric_cols})
    tipo = pd.concat(parciales, ignore_index=True)[KEYS + ['TIPOSEC']].drop_duplicates(KEYS)
    result = result.merge(tipo, on=KEYS, how='left')
    total = (result['SV'] + result['NV']).replace(0, np.nan)
    nominal = result['LN'].replace(0, np.nan)
    result['tasa_abstencion'] = result['NV'] / total
    result['tasa_participacion'] = result['SV'] / total
    result['ns_ratio'] = result['NS'] / nominal
    for name in ['18_29', '30_59', '60_mas', 'hombres', 'mujeres', 'no_binario']:
        result[f'prop_{name}'] = result[name] / nominal
    return result
elecciones = {year: agregar_eleccion(year) for year in YEARS}
print({year: len(frame) for year, frame in elecciones.items()})


{2009: 64934, 2012: 65599, 2015: 68362, 2018: 68408, 2021: 68812, 2024: 70751}


In [3]:
historical_features = ['tasa_abstencion', 'tasa_participacion', 'ns_ratio', 'LN', 'prop_18_29', 'prop_30_59', 'prop_60_mas', 'prop_hombres', 'prop_mujeres', 'prop_no_binario', 'TIPOSEC']
data = elecciones[2024][KEYS + ['tasa_abstencion', 'SV', 'NV', 'LN']].rename(columns={'tasa_abstencion': 'abstencion_2024', 'SV': 'SV_2024', 'NV': 'NV_2024', 'LN': 'LN_2024'}).copy()
for year in YEARS[:-1]:
    frame = elecciones[year][KEYS + historical_features].rename(columns={col: f'{col}_{year}' for col in historical_features})
    data = data.merge(frame, on=KEYS, how='left')
print('Secciones con objetivo 2024:', len(data))
numeric_features = [f'{col}_{year}' for year in YEARS[:-1] for col in historical_features if col != 'TIPOSEC']
categorical_features = [f'TIPOSEC_{year}' for year in YEARS[:-1]]
model_features = numeric_features + categorical_features
eps = 1e-4
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), eps, 1 - eps)
    return np.log(p / (1 - p))
def inv_logit(x): return 1 / (1 + np.exp(-x))
data = data.dropna(subset=['abstencion_2024']).copy()
data['objetivo_logit'] = logit(data['abstencion_2024'])


Secciones con objetivo 2024: 70751


In [4]:
preprocessor = ColumnTransformer(transformers=[('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features), ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)])
def crear_modelo():
    return Pipeline([('preprocessor', preprocessor), ('regressor', HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0, random_state=42))])
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(data, groups=data['MPIOCVE'].astype(str) + '_' + data['EDOCVE'].astype(str)))
model_eval = crear_modelo()
model_eval.fit(data.iloc[train_idx][model_features], data.iloc[train_idx]['objetivo_logit'])
pred_test = np.clip(inv_logit(model_eval.predict(data.iloc[test_idx][model_features])), 0, 1)
actual_test = data.iloc[test_idx]['abstencion_2024']
baseline_test = data.iloc[test_idx]['tasa_abstencion_2021'].fillna(data.iloc[test_idx]['tasa_abstencion_2021'].median()).clip(0, 1)
metricas = pd.DataFrame({'modelo': ['Gradient Boosting con historia 2009–2021', 'Baseline: abstención 2021'], 'MAE': [mean_absolute_error(actual_test, pred_test), mean_absolute_error(actual_test, baseline_test)], 'RMSE': [mean_squared_error(actual_test, pred_test) ** 0.5, mean_squared_error(actual_test, baseline_test) ** 0.5], 'R2': [r2_score(actual_test, pred_test), r2_score(actual_test, baseline_test)]})
print(f'Secciones de entrenamiento: {len(train_idx):,}')
print(f'Secciones de prueba: {len(test_idx):,}')
print(metricas)
model = crear_modelo()
model.fit(data[model_features], data['objetivo_logit'])
data['riesgo_modelo'] = np.clip(inv_logit(model.predict(data[model_features])), 0, 1)
data['riesgo_baseline'] = data['tasa_abstencion_2021'].fillna(data['tasa_abstencion_2021'].median()).clip(0, 1)


Secciones de entrenamiento: 56,907
Secciones de prueba: 11,744
                                     modelo       MAE      RMSE        R2
0  Gradient Boosting con historia 2009–2021  0.041236  0.058825  0.681025
1                 Baseline: abstención 2021  0.090001  0.114299 -0.204247


## Evaluación

La validación reserva municipios completos. El modelo final usa todas las secciones 2024 únicamente después de medir el desempeño, para producir el índice operativo.


In [5]:
data['riesgo'] = (100 * data['riesgo_modelo']).round(2)
data['nivel_riesgo'] = pd.cut(data['riesgo'], bins=[-np.inf, 35, 50, 65, np.inf], labels=['Bajo', 'Medio', 'Alto', 'Muy alto'])
data['abstencion_esperada_personas'] = data['riesgo_modelo'] * data['LN_2024']
data['prioridad'] = data['abstencion_esperada_personas']
resumen_niveles = data.groupby('nivel_riesgo', observed=False).agg(secciones=('nivel_riesgo', 'size'), lista_nominal=('LN_2024', 'sum'), abstencion_esperada=('abstencion_esperada_personas', 'sum'), riesgo_promedio=('riesgo', 'mean')).reset_index()
print(resumen_niveles)


  nivel_riesgo  secciones  lista_nominal  abstencion_esperada  riesgo_promedio
0         Bajo      22865       28570890         8.480919e+06        29.337665
1        Medio      37433       54528983         2.295627e+07        41.921224
2         Alto       8066       13183858         7.171935e+06        54.259466
3     Muy alto        287         457975         3.107107e+05        68.805679


## Simulación de escenarios

- **X1 informativa:** reducción hipotética de 3 puntos porcentuales en la abstención estimada de secciones Alto/Muy alto.
- **X2 intensiva:** reducción hipotética de 7 puntos porcentuales en esas mismas secciones.

Son análisis de sensibilidad, no efectos causales estimados. Una intervención real requeriría medición antes/después, grupo de comparación o una prueba controlada.


In [6]:
intervencion = data['nivel_riesgo'].isin(['Alto', 'Muy alto'])
for nombre, reduccion in [('X1_informativa', 0.03), ('X2_intensiva', 0.07)]:
    tasa = data['riesgo_modelo'].where(~intervencion, (data['riesgo_modelo'] - reduccion).clip(lower=0))
    data[nombre] = tasa
    data[f'personas_movilizables_{nombre}'] = (data['riesgo_modelo'] - tasa) * data['LN_2024']
escenarios = pd.DataFrame({'escenario': ['Sin intervención', 'X1 informativa', 'X2 intensiva'], 'reduccion_aplicada': ['0 puntos', '3 puntos', '7 puntos'], 'personas_potencialmente_movilizables': [0, data['personas_movilizables_X1_informativa'].sum(), data['personas_movilizables_X2_intensiva'].sum()], 'abstencion_promedio_escenario': [data['riesgo_modelo'].mean(), data['X1_informativa'].mean(), data['X2_intensiva'].mean()]})
print(escenarios)


          escenario reduccion_aplicada  personas_potencialmente_movilizables  \
0  Sin intervención           0 puntos                                  0.00   
1    X1 informativa           3 puntos                             409254.99   
2      X2 intensiva           7 puntos                             954928.31   

   abstencion_promedio_escenario  
0                       0.392920  
1                       0.389270  
2                       0.384403  


In [7]:
resultado_final = data.sort_values(['nivel_riesgo', 'prioridad'], ascending=[True, False]).copy()
resultado_final.to_csv(BASE_DIR / 'resultado_modelo_riesgo_2024.csv', index=False)
escenarios.to_csv(BASE_DIR / 'escenarios_modelo_riesgo_2024.csv', index=False)
print('Archivos generados:', BASE_DIR / 'resultado_modelo_riesgo_2024.csv', BASE_DIR / 'escenarios_modelo_riesgo_2024.csv')


Archivos generados: /Users/marcosbautista/Documents/GitHub/Reto2_INE/resultado_modelo_riesgo_2024.csv /Users/marcosbautista/Documents/GitHub/Reto2_INE/escenarios_modelo_riesgo_2024.csv


## Interpretación y límites

- Se incorporan los datos históricos 2009, 2012, 2015, 2018 y 2021 como predictores del resultado 2024.
- El riesgo es territorial, no una probabilidad individual.
- `NS` es indicador de calidad, no voto emitido ni abstención.
- Las claves de sección pueden cambiar entre elecciones; debe verificarse la comparabilidad cartográfica.
- X1 y X2 son escenarios hipotéticos; no representan impacto observado.


## Resultados de la versión canónica
La versión reproducible en `src/entrenar_modelo.py` incorpora contexto espacial municipal/estatal y comparación de modelos con tres semillas. Resultados de la ejecución actual:
- Modelo final: `hgb_espacial` (HistGradientBoosting + historia 2009–2021 + contexto espacial).
- MAE: **0.0323 ± 0.0018**; RMSE: **0.0473 ± 0.0023**; R²: **0.784 ± 0.011**.
- Línea base 2021: MAE **0.0974 ± 0.0038** (reducción ≈ 66.9%).
- 70,751 secciones en el entregable; 68,651 modelables y 2,100 marcadas como `Sin dato`.
- 9,641 secciones en riesgo alto/muy alto; X1 moviliza 477,315 personas y X2 1,113,735.
- Validación temporal (entrenar ≤2018, predecir 2021): MAE **0.0721** vs línea base **0.1001**.
- Intervalos de predicción del 80%: cobertura empírica **80.0%** y ancho medio **10.1 puntos**.
- Ranking estatal con rangos de incertidumbre y alertas por secciones sin dato en `resumen_entidades.csv`.
Este notebook conserva la lógica exploratoria original; para las cifras oficiales del entregable usa el script y los CSV de `DatosAbiertos/processed/`.